# Molecule Regression Workflow with ESOL Example

This notebook demonstrates a complete molecule regression workflow using **kgcnn_torch** (PyTorch)
with the **DMPNN** (Directed Message Passing Neural Network) model.

Steps:
1. Load and preprocess the ESOL dataset using `MoleculeNetDataset`
2. Preprocess with `SetRange` and `SetEdgeIndicesReverse`
3. Convert molecular graphs to PyG Data objects
4. Build a DMPNN model
5. Train with cross-validation using `trainer.fit()`
6. Evaluate and plot results

In [ ]:
import os
import torch
import numpy as np
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## 1. Load the ESOL Dataset

We use `MoleculeNetDataset` to load the ESOL solubility dataset. The dataset is
expected to be in a local directory with a CSV file and SDF structures.

To download the data, use the kgcnn-torch `ESOLDataset` or Keras `ESOLDataset`:
```python
from kgcnn_torch.data.datasets.ESOLDataset import ESOLDataset
ESOLDataset()
```
Then copy the data to a local `esol/` folder, or the dataset will be downloaded
automatically.

In [ ]:
from kgcnn_torch.data.moleculenet import MoleculeNetDataset

data = MoleculeNetDataset(
    data_directory="esol/",
    dataset_name="esol",
    file_name="delaney-processed.csv",
)

## 2. Prepare Data: Generate Conformers from SMILES

This converts SMILES to 3D molecular structures stored as SDF files.

In [ ]:
data.prepare_data(
    overwrite=False,
    smiles_column_name="smiles",
    add_hydrogen=True,
    sanitize=True,
    make_conformers=True,
    optimize_conformer=True,
    external_program=None,
    num_workers=None
)

## 3. Read Into Memory: Extract Node/Edge Features

We extract molecular features using RDKit: atom types (one-hot encoded), bond types, etc.

In [ ]:
from kgcnn_torch.molecule.encoder import OneHotEncoder

data.read_in_memory(
    nodes=[
        'Symbol', 'TotalDegree', 'FormalCharge', 'NumRadicalElectrons', 'Hybridization',
        'IsAromatic', 'IsInRing', 'TotalNumHs', 'CIPCode', 'ChiralityPossible', 'ChiralTag'
    ],
    encoder_nodes={
        'Symbol': OneHotEncoder(
            ['B', 'C', 'N', 'O', 'F', 'Si', 'P', 'S', 'Cl', 'As', 'Se', 'Br', 'Te', 'I', 'At'],
            dtype="str"
        ),
        'Hybridization': OneHotEncoder([2, 3, 4, 5, 6]),
        'TotalDegree': OneHotEncoder([0, 1, 2, 3, 4, 5], add_unknown=False),
        'TotalNumHs': OneHotEncoder([0, 1, 2, 3, 4], add_unknown=False),
        'CIPCode': OneHotEncoder(['R', 'S'], add_unknown=False, dtype='str'),
        'ChiralityPossible': OneHotEncoder(['1'], add_unknown=False, dtype='str'),
    },
    edges=['BondType', 'IsAromatic', 'IsConjugated', 'IsInRing', 'Stereo'],
    encoder_edges={
        'BondType': OneHotEncoder([1, 2, 3, 12], add_unknown=False),
        'Stereo': OneHotEncoder([0, 1, 2, 3], add_unknown=False)
    },
    graph=['ExactMolWt', 'NumAtoms'],
    encoder_graph={},
    add_hydrogen=False,
    make_directed=False,
    has_conformers=True,
    sanitize=True,
    compute_partial_charges=None,
    label_column_name="measured log solubility in mols per litre"
)

print("Number of graphs:", len(data))
print("First graph keys:", data[0].keys())

## 4. Preprocessing for DMPNN

DMPNN requires:
- `SetEdgeIndicesReverse`: Map each directed edge to its reverse counterpart
- `count_nodes_and_edges`: Count totals for batching

These preprocessing steps match the Keras workflow notebook.

In [ ]:
from kgcnn_torch.graph.preprocessor import SetEdgeIndicesReverse

data.map_list(SetEdgeIndicesReverse(in_place=True))
data.map_list(method="count_nodes_and_edges")

# Rename edge_indices_reverse -> edge_pair_index for PyG compatibility
for g in data:
    reverse = g.obtain_property("edge_indices_reverse")
    if reverse is not None:
        g["edge_pair_index"] = reverse

print("Updated keys:", data[0].keys())

## 5. Clean Invalid Graphs and Extract Labels

In [ ]:
# Remove graphs with missing edge indices
removed = data.clean("edge_indices")
print("Removed graphs:", removed)
print("Remaining graphs:", len(data))

# Extract labels
labels = np.array(data.obtain_property("graph_labels"))
if len(labels.shape) <= 1:
    labels = np.expand_dims(labels, axis=-1)
print("Labels shape:", labels.shape)

## 6. Convert to PyG Data Objects

The `to_pyg_list()` method converts the KGCNN MemoryGraphList into PyTorch Geometric Data objects,
automatically swapping the edge index convention (KGCNN: target,source -> PyG: source,target).

We also squeeze `edge_pair_index` from (M, 1) to (M,) as required by the DMPNN model.

In [ ]:
pyg_list = data.to_pyg_list()

# Squeeze edge_pair_index from (M, 1) to (M,) for DMPNN
for d in pyg_list:
    if hasattr(d, 'edge_pair_index') and d.edge_pair_index is not None:
        if d.edge_pair_index.dim() == 2:
            d.edge_pair_index = d.edge_pair_index.squeeze(-1)

print("First PyG data object:")
print(pyg_list[0])
print("Node features shape:", pyg_list[0].x.shape)
print("Edge index shape:", pyg_list[0].edge_index.shape)
print("Edge attr shape:", pyg_list[0].edge_attr.shape)
print("Edge pair index shape:", pyg_list[0].edge_pair_index.shape)
print("Label:", pyg_list[0].y)

## 7. Cross-Validation Split

We use 5-fold cross-validation to evaluate the model.

In [ ]:
from sklearn.model_selection import KFold

kf = KFold(n_splits=5, random_state=42, shuffle=True)
train_test_indices = [
    [train_index, test_index]
    for train_index, test_index in kf.split(X=np.zeros((len(pyg_list), 1)), y=labels)
]
print("Number of folds:", len(train_test_indices))
print("Fold 0 train/test sizes:", len(train_test_indices[0][0]), len(train_test_indices[0][1]))

## 8. Define Model: DMPNN for Regression

We use `DMPNNModel` from kgcnn_torch. DMPNN performs directed message passing on edges,
using `edge_pair_index` to exclude reverse-edge messages at each step.

This matches the Keras workflow notebook which also uses DMPNN.

In [ ]:
from kgcnn_torch.models.dmpnn import DMPNNModel

# Determine input dimensions from data
node_input_dim = pyg_list[0].x.shape[1]
edge_dim = pyg_list[0].edge_attr.shape[1]
print("Node input dimension:", node_input_dim)
print("Edge input dimension:", edge_dim)

# Model configuration matching the Keras DMPNN workflow
model_kwargs = dict(
    node_dim=64,
    edge_dim=edge_dim,
    depth=5,
    units=128,
    message_activation="relu",
    node_pooling="sum",
    output_units=[64, 32],
    output_activation="relu",
    num_targets=1,
    output_embedding="graph",
    use_node_embedding=False,
    node_input_dim=node_input_dim,
    dropout_rate=0.1,
)

## 9. Training Loop with Cross-Validation

For each fold:
1. Create a fresh model
2. Normalize labels with `StandardLabelScaler`
3. Create PyG DataLoaders
4. Train using `kgcnn_torch.training.trainer.fit()`
5. Collect history for plotting

In [ ]:
import time
from datetime import timedelta
from torch_geometric.loader import DataLoader
from kgcnn_torch.training.trainer import fit
from kgcnn_torch.data.transform import StandardLabelScaler

history_list = []
test_indices_list = []
model = None
scaler = None

for fold_idx, (train_index, test_index) in enumerate(train_test_indices):
    print(f"\n=== Fold {fold_idx} ===")

    # Create fresh model for each fold
    model = DMPNNModel(**model_kwargs)
    model = model.to(device)

    # Split data
    train_data = [pyg_list[i] for i in train_index]
    test_data = [pyg_list[i] for i in test_index]

    # Fit scaler on training labels
    y_train = labels[train_index]
    y_test = labels[test_index]

    scaler = StandardLabelScaler()
    scaler.fit(y_train)
    y_train_scaled = scaler.transform(y_train)
    y_test_scaled = scaler.transform(y_test)

    # Update labels in PyG data objects (use scaled labels)
    for i, idx in enumerate(train_index):
        train_data[i].y = torch.tensor(y_train_scaled[i], dtype=torch.float)
    for i, idx in enumerate(test_index):
        test_data[i].y = torch.tensor(y_test_scaled[i], dtype=torch.float)

    # Create DataLoaders
    train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
    test_loader = DataLoader(test_data, batch_size=32, shuffle=False)

    # Define optimizer, loss, scheduler
    optimizer = torch.optim.Adam(model.parameters(), lr=5e-4)
    loss_fn = torch.nn.L1Loss()  # MAE loss
    scheduler = torch.optim.lr_scheduler.LinearLR(
        optimizer, start_factor=1.0, end_factor=0.01, total_iters=300
    )

    # Define metrics in original scale
    def mae_metric(pred, target):
        return torch.mean(torch.abs(pred - target))

    def rmse_metric(pred, target):
        return torch.sqrt(torch.mean((pred - target) ** 2))

    metrics = {"mae": mae_metric, "rmse": rmse_metric}

    # Train
    start = time.process_time()
    history = fit(
        model=model,
        train_loader=train_loader,
        val_loader=test_loader,
        optimizer=optimizer,
        loss_fn=loss_fn,
        scheduler=scheduler,
        epochs=300,
        device=device,
        metrics=metrics,
        verbose=1,
        scaler=scaler,
    )
    stop = time.process_time()
    print(f"Training time: {timedelta(seconds=stop - start)}")

    history_list.append(history)
    test_indices_list.append([train_index, test_index])

## 10. Plot Training Curves

We use the kgcnn_torch plotting utility to visualize training and validation loss across folds.

In [ ]:
from kgcnn_torch.utils.plots import plot_train_test_loss, plot_predict_true

plot_train_test_loss(
    history_list,
    loss_name="train_loss",
    val_loss_name="val_loss",
    model_name="DMPNN",
    data_unit="mol/L",
    dataset_name="ESOL",
    filepath=None,
    file_name="loss.png",
);

## 11. Prediction vs True Plot (Last Fold)

We run inference on the test set from the last fold, inverse-transform the predictions,
and make a scatter plot.

In [ ]:
# Get predictions for the last fold's test set
model.eval()
last_train_idx, last_test_idx = test_indices_list[-1]
test_data_last = [pyg_list[i] for i in last_test_idx]
test_loader_last = DataLoader(test_data_last, batch_size=32, shuffle=False)

all_preds = []
with torch.no_grad():
    for batch in test_loader_last:
        batch = batch.to(device)
        pred = model(batch)
        all_preds.append(pred.cpu().numpy())

predicted_y = np.concatenate(all_preds, axis=0)
true_y_scaled = labels[last_test_idx]

# Inverse transform to original scale
predicted_y_orig = scaler.inverse_transform(predicted_y)
true_y_orig = true_y_scaled  # labels were never scaled in the labels array

plot_predict_true(
    predicted_y_orig, true_y_orig,
    data_unit="mol/L",
    model_name="DMPNN",
    dataset_name="ESOL",
    filepath=None,
    file_name="predict.png",
    show_fig=True,
);

## Alternative: Using GIN or GCN Model

You can easily swap the DMPNN model for GIN or GCN by changing the import and configuration.
Note that GIN and GCN do not require `edge_pair_index` (reverse edges).

In [ ]:
from kgcnn_torch.models.gin import GINModel

gin_model = GINModel(
    node_dim=64,
    depth=3,
    units=64,
    gin_mlp_units=[64, 64],
    gin_mlp_activation="relu",
    gin_pooling="sum",
    node_pooling="sum",
    output_units=[],
    output_final_activation="linear",
    num_targets=1,
    output_embedding="graph",
    use_node_embedding=False,
    node_input_dim=node_input_dim,
)

print(gin_model)
print(f"GIN model parameters: {sum(p.numel() for p in gin_model.parameters()):,}")